# Single-Cell Z-Score Analysis for Protein Abundance Change

This notebook implements the DUAL-IPA-inspired single-cell Z-score methodology for protein abundance analysis.

## Key Differences from Well-Aggregated Approach (2_protein_abundance_change_hits.ipynb)

| Aspect | Previous (Well-Aggregated) | This (Single-Cell Z-Score) |
|--------|---------------------------|---------------------------|
| Unit of analysis | Well median | Individual cell |
| Statistical test | Paired t-test | Z-score normalization |
| Reference handling | Well-level pairing | Cell population distribution |
| Heterogeneity | Lost | Preserved |

In [ ]:
### Imports
import os
import polars as pl
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
sys.path.append("../..")
from img_utils import *
%matplotlib inline

# GFP feature to use (MeanIntensity is normalized by cell area)
GFP_FEAT = "Cells_Intensity_MeanIntensity_GFP"

## Configuration & QC Thresholds

In [ ]:
# QC thresholds (relaxed to maximize variant coverage while maintaining statistical validity)
# See docs/single_cell_zscore_methodology.md for rationale
MIN_CELLS_PER_WELL = 20   # Minimum cells per well for statistical reliability
MIN_REF_CELLS = 20        # Minimum reference cells for std estimation
MIN_REF_WELLS_PER_PLATE = 1  # Minimum reference wells per gene/plate

# Output directory
OUTPUT_DIR = CC_ABUND_DIR

## 1. Load Metadata

### 1.1 Load Plate Background Data (GFP channel)

In [ ]:
# Load image well QC summary
img_well_qc_sum_df = pl.read_parquet(IMG_QC_SUM_PARQUET_FILE)

# Extract GFP channel plate background and is_bg flag
plate_bg_df = (
    img_well_qc_sum_df
    .filter(pl.col("channel") == "GFP")
    .select(["plate", "well", "median_plate", "is_bg", "gene_allele", "node_type", "Metadata_Bio_Batch"])
    .unique()
    .rename({"plate": "Metadata_Plate", "well": "Metadata_Well"})
)

print(f"Loaded plate background for {plate_bg_df.select('Metadata_Plate').n_unique()} plates")
print(f"Background wells: {plate_bg_df.filter(pl.col('is_bg')).shape[0]}")
print(f"Non-background wells: {plate_bg_df.filter(~pl.col('is_bg')).shape[0]}")
plate_bg_df.head()

### 1.2 Get list of tested variants

In [ ]:
# Get tested variants (allele type, not reference/control)
tested_variants = (
    img_well_qc_sum_df
    .filter(
        (pl.col("gene_allele").is_not_null()) & 
        (pl.col("node_type") == "allele")
    )
    .unique(subset=["gene_allele"])["gene_allele"]
)
print(f"Number of tested variants: {len(tested_variants)}")

## 2. Load Single-Cell GFP Profiles

Load GFP intensity features from single-cell profiles across all batches.

In [ ]:
# Load single-cell GFP profiles from all batches
all_sc_profiles = []

for bio_rep, bio_rep_batches in tqdm(BIO_REP_BATCHES_DICT.items(), desc="Loading batches"):
    for batch_id in bio_rep_batches:
        profile_path = f"{PROF_DIR}/{batch_id}/profiles.parquet"
        
        if not os.path.exists(profile_path):
            print(f"Warning: {profile_path} not found, skipping")
            continue
        
        # Load profiles with relevant columns
        batch_profiles = (
            pl.scan_parquet(profile_path)
            .select([
                "Metadata_Plate",
                "Metadata_Well",
                "Metadata_ImageNumber",
                "Metadata_ObjectNumber",
                "Metadata_gene_allele",
                GFP_FEAT
            ])
            .with_columns(
                pl.concat_str(
                    ["Metadata_Plate", "Metadata_Well", "Metadata_ImageNumber", "Metadata_ObjectNumber"],
                    separator="_"
                ).alias("Metadata_CellID"),
                pl.lit(batch_id).alias("Metadata_Batch")
            )
            .collect()
        )
        
        all_sc_profiles.append(batch_profiles)

# Combine all profiles
sc_profiles = pl.concat(all_sc_profiles)
print(f"Total single cells loaded: {sc_profiles.shape[0]:,}")

### 2.1 Join with well QC data and filter\n\n**Note on background handling:** The CellProfiler GFP intensities are already normalized to 0-1 scale during the segmentation pipeline. We do NOT subtract the raw image `median_plate` value (which is in different units). Instead, background is handled by:\n1. Filtering out wells flagged as `is_bg=True` (background-only wells identified during image QC)\n2. The Z-score methodology itself, which normalizes variant cells relative to reference cells on the same plate

In [ ]:
# Join plate background to single-cell data (for is_bg flag and bio_batch info)
sc_profiles = sc_profiles.join(
    plate_bg_df.select(["Metadata_Plate", "Metadata_Well", "is_bg", "Metadata_Bio_Batch"]),
    on=["Metadata_Plate", "Metadata_Well"],
    how="left"
)

print(f"Cells before filtering: {sc_profiles.shape[0]:,}")

# Filter out background-only wells (identified via image-level QC)
sc_profiles = sc_profiles.filter(~pl.col("is_bg").fill_null(True))
print(f"Cells after removing background wells: {sc_profiles.shape[0]:,}")

# NOTE: CellProfiler GFP intensities are already normalized (0-1 scale)
# We do NOT subtract the raw image median_plate (which is in raw image units ~150)
# Instead, we use the CellProfiler values directly for the Z-score calculation
# The reference distribution approach inherently handles background by comparing
# variant cells to reference cells on the same plate.

# Rename GFP feature to corrected_GFP for consistency with later code
sc_profiles = sc_profiles.with_columns(
    pl.col(GFP_FEAT).alias("corrected_GFP")
)

# Filter out cells with zero or very low GFP (likely segmentation artifacts)
min_gfp_threshold = 1e-6
n_before = sc_profiles.shape[0]
sc_profiles = sc_profiles.filter(pl.col("corrected_GFP") > min_gfp_threshold)
n_removed = n_before - sc_profiles.shape[0]
print(f"Removed {n_removed:,} cells with GFP <= {min_gfp_threshold}")

print(f"\\nGFP intensity stats after filtering:")
print(f"  Min: {sc_profiles['corrected_GFP'].min():.6f}")
print(f"  Max: {sc_profiles['corrected_GFP'].max():.6f}")
print(f"  Mean: {sc_profiles['corrected_GFP'].mean():.6f}")
print(f"  Median: {sc_profiles['corrected_GFP'].median():.6f}")

## 3. Compute Reference Statistics

For each gene and plate, compute the reference distribution parameters:
- μ_ref: mean reference intensity
- μ_WT: mean log2FC of reference cells relative to μ_ref (should be ~0)
- σ_WT: standard deviation of reference log2FC

In [ ]:
def compute_ref_stats_for_gene_plate(sc_df: pl.DataFrame, gene: str, plate: str):
    """
    Compute reference distribution statistics for a gene on a plate.
    
    Returns:
        dict with keys: mu_ref, mu_wt_log2fc, sigma_wt_log2fc, n_ref_cells, n_ref_wells
        or None if insufficient data
    """
    # Filter for reference allele (gene name without variant suffix)
    ref_cells = sc_df.filter(
        (pl.col("Metadata_gene_allele") == gene) &
        (pl.col("Metadata_Plate") == plate)
    )
    
    if ref_cells.height < MIN_REF_CELLS:
        return None
    
    # Check minimum wells
    n_ref_wells = ref_cells.select("Metadata_Well").n_unique()
    if n_ref_wells < MIN_REF_WELLS_PER_PLATE:
        return None
    
    # Mean reference intensity
    mu_ref = ref_cells["corrected_GFP"].mean()
    
    # Log2FC for each reference cell relative to mean
    ref_log2fc = np.log2(ref_cells["corrected_GFP"].to_numpy() / mu_ref)
    
    # Distribution parameters
    mu_wt_log2fc = np.mean(ref_log2fc)
    sigma_wt_log2fc = np.std(ref_log2fc, ddof=1)
    
    return {
        "mu_ref": mu_ref,
        "mu_wt_log2fc": mu_wt_log2fc,
        "sigma_wt_log2fc": sigma_wt_log2fc,
        "n_ref_cells": ref_cells.height,
        "n_ref_wells": n_ref_wells
    }

In [ ]:
# Pre-compute reference statistics for all gene-plate combinations
# First, get all unique genes (reference alleles)
all_genes = (
    sc_profiles
    .filter(pl.col("Metadata_gene_allele").is_not_null())
    .select("Metadata_gene_allele")
    .unique()
    .with_columns(
        pl.col("Metadata_gene_allele").str.split("_").list.first().alias("gene")
    )
    # Reference alleles are those where gene_allele == gene (no variant suffix)
    .filter(pl.col("Metadata_gene_allele") == pl.col("gene"))
    ["gene"].unique().to_list()
)

print(f"Number of reference genes: {len(all_genes)}")

In [ ]:
# Compute reference statistics for each gene-plate combination
ref_stats_cache = {}

for gene in tqdm(all_genes, desc="Computing reference stats"):
    # Get plates where this gene appears
    gene_plates = (
        sc_profiles
        .filter(pl.col("Metadata_gene_allele") == gene)
        .select("Metadata_Plate")
        .unique()["Metadata_Plate"].to_list()
    )
    
    for plate in gene_plates:
        stats = compute_ref_stats_for_gene_plate(sc_profiles, gene, plate)
        if stats is not None:
            ref_stats_cache[(gene, plate)] = stats

print(f"Computed reference stats for {len(ref_stats_cache)} gene-plate combinations")

## 4. Compute Variant Z-Scores

For each variant cell:
```
Z_cell = (log2(variant_cell_intensity / μ_ref) - μ_WT) / σ_WT
```

In [ ]:
def compute_variant_zscores(sc_df: pl.DataFrame, variant: str, gene: str, plate: str, ref_stats: dict):
    """
    Compute Z-scores for all cells of a variant on a plate.
    
    Returns:
        DataFrame with cell-level Z-scores, or None if insufficient data
    """
    mu_ref = ref_stats["mu_ref"]
    mu_wt = ref_stats["mu_wt_log2fc"]
    sigma_wt = ref_stats["sigma_wt_log2fc"]
    
    # Avoid division by zero
    if sigma_wt == 0 or np.isnan(sigma_wt):
        return None
    
    var_cells = sc_df.filter(
        (pl.col("Metadata_gene_allele") == variant) &
        (pl.col("Metadata_Plate") == plate)
    )
    
    if var_cells.height == 0:
        return None
    
    # Log2FC relative to reference mean
    var_log2fc = np.log2(var_cells["corrected_GFP"].to_numpy() / mu_ref)
    
    # Z-score normalized by WT variability
    z_scores = (var_log2fc - mu_wt) / sigma_wt
    
    return var_cells.with_columns(
        pl.Series("log2FC", var_log2fc),
        pl.Series("zscore", z_scores),
        pl.lit(gene).alias("Gene"),
        pl.lit(ref_stats["n_ref_cells"]).alias("n_ref_cells"),
        pl.lit(mu_ref).alias("mu_ref"),
        pl.lit(sigma_wt).alias("sigma_wt")
    )

In [ ]:
# Compute Z-scores for all variants
all_variant_zscores = []

for variant in tqdm(tested_variants, desc="Computing variant Z-scores"):
    # Extract gene from variant name
    gene = variant.split("_")[0]
    
    # Skip if this is a reference allele
    if gene == variant:
        continue
    
    # Get plates where this variant appears
    variant_plates = (
        sc_profiles
        .filter(pl.col("Metadata_gene_allele") == variant)
        .select("Metadata_Plate")
        .unique()["Metadata_Plate"].to_list()
    )
    
    for plate in variant_plates:
        # Check if we have reference stats for this gene-plate
        if (gene, plate) not in ref_stats_cache:
            continue
        
        ref_stats = ref_stats_cache[(gene, plate)]
        
        # Compute Z-scores
        var_zscores = compute_variant_zscores(sc_profiles, variant, gene, plate, ref_stats)
        
        if var_zscores is not None:
            all_variant_zscores.append(var_zscores)

# Combine all variant Z-scores
variant_zscores_df = pl.concat(all_variant_zscores) if all_variant_zscores else pl.DataFrame()
print(f"Total variant cells with Z-scores: {variant_zscores_df.shape[0]:,}")

## 5. Hierarchical Aggregation

1. Per-plate aggregation: mean(Z_cell) for all variant cells on plate
2. Cross-plate aggregation: median(Z_variant_plate) across plates

In [ ]:
# Step 1: Mean aggregate Z-scores per variant per plate
plate_zscores = (
    variant_zscores_df
    .group_by(["Metadata_gene_allele", "Gene", "Metadata_Plate", "Metadata_Bio_Batch"])
    .agg([
        pl.col("zscore").mean().alias("mean_zscore"),
        pl.col("zscore").std().alias("std_zscore"),
        pl.col("zscore").median().alias("median_zscore"),
        pl.col("log2FC").mean().alias("mean_log2FC"),
        pl.len().alias("n_cells"),
        pl.col("n_ref_cells").first().alias("n_ref_cells")
    ])
)

print(f"Plate-level Z-scores: {plate_zscores.shape[0]} entries")
plate_zscores.head()

In [ ]:
# Step 2: Median aggregate across plates (within bio batch)
batch_zscores = (
    plate_zscores
    .group_by(["Metadata_gene_allele", "Gene", "Metadata_Bio_Batch"])
    .agg([
        pl.col("mean_zscore").median().alias("final_zscore"),
        pl.col("mean_zscore").mean().alias("mean_of_plate_zscores"),
        pl.col("mean_zscore").std().alias("zscore_plate_variability"),
        pl.col("mean_log2FC").median().alias("median_log2FC"),
        pl.col("n_cells").sum().alias("total_cells"),
        pl.col("n_ref_cells").mean().alias("avg_ref_cells_per_plate"),
        pl.len().alias("n_plates")
    ])
)

print(f"Bio-batch level Z-scores: {batch_zscores.shape[0]} entries")
batch_zscores.head()

## 6. P-Value Calculation

### Method 1: Parametric (Normal CDF)
Two-tailed p-value from Z-score assuming normal distribution.

### Method 2: Empirical (WT-to-WT Distribution)
Compute Z-scores for reference wells treating each as a "variant" and use this as the null distribution.

In [ ]:
def compute_wt_to_wt_zscores(sc_df: pl.DataFrame, ref_stats_cache: dict):
    """
    Compute Z-scores for reference wells using leave-one-out approach.
    For each reference well, compute its Z-score relative to other reference wells.
    
    Returns:
        Array of WT-to-WT Z-scores for empirical null distribution
    """
    wt_zscores = []
    
    for (gene, plate), ref_stats in ref_stats_cache.items():
        # Get reference cells on this plate
        ref_cells = sc_df.filter(
            (pl.col("Metadata_gene_allele") == gene) &
            (pl.col("Metadata_Plate") == plate)
        )
        
        if ref_cells.height < MIN_REF_CELLS:
            continue
        
        # Get unique wells
        wells = ref_cells.select("Metadata_Well").unique()["Metadata_Well"].to_list()
        
        for well in wells:
            # Get cells from this well
            well_cells = ref_cells.filter(pl.col("Metadata_Well") == well)
            
            if well_cells.height < MIN_CELLS_PER_WELL:
                continue
            
            # Compute Z-score for this well's cells
            mu_ref = ref_stats["mu_ref"]
            mu_wt = ref_stats["mu_wt_log2fc"]
            sigma_wt = ref_stats["sigma_wt_log2fc"]
            
            if sigma_wt == 0 or np.isnan(sigma_wt):
                continue
            
            # Mean log2FC for this well
            well_log2fc = np.log2(well_cells["corrected_GFP"].to_numpy() / mu_ref)
            well_mean_log2fc = np.mean(well_log2fc)
            
            # Z-score
            well_zscore = (well_mean_log2fc - mu_wt) / sigma_wt
            wt_zscores.append(well_zscore)
    
    return np.array(wt_zscores)

# Compute WT-to-WT Z-scores
wt_zscores = compute_wt_to_wt_zscores(sc_profiles, ref_stats_cache)
print(f"Number of WT-to-WT Z-scores: {len(wt_zscores)}")
print(f"Mean: {np.mean(wt_zscores):.4f}, Std: {np.std(wt_zscores):.4f}")

In [ ]:
# Visualize WT-to-WT distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(wt_zscores, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
x = np.linspace(-4, 4, 100)
axes[0].plot(x, norm.pdf(x), 'r-', lw=2, label='Standard Normal')
axes[0].set_xlabel('WT-to-WT Z-score')
axes[0].set_ylabel('Density')
axes[0].set_title('WT-to-WT Z-score Distribution')
axes[0].legend()

# QQ plot
from scipy import stats
stats.probplot(wt_zscores, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot vs Standard Normal')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate p-values
def calculate_pvalues(zscore, wt_zscores):
    """
    Calculate both parametric and empirical p-values.
    
    Returns:
        (pval_parametric, pval_empirical)
    """
    # Parametric: two-tailed p-value from normal distribution
    pval_parametric = 2 * norm.sf(np.abs(zscore))
    
    # Empirical: proportion of WT Z-scores more extreme
    pval_empirical = np.mean(np.abs(wt_zscores) >= np.abs(zscore))
    
    # Floor empirical p-value to avoid p=0
    if pval_empirical == 0:
        pval_empirical = 1.0 / (len(wt_zscores) + 1)
    
    return pval_parametric, pval_empirical

# Apply to all variants
pvals_parametric = []
pvals_empirical = []

for zscore in batch_zscores["final_zscore"].to_list():
    if zscore is None or np.isnan(zscore):
        pvals_parametric.append(np.nan)
        pvals_empirical.append(np.nan)
    else:
        p_param, p_emp = calculate_pvalues(zscore, wt_zscores)
        pvals_parametric.append(p_param)
        pvals_empirical.append(p_emp)

batch_zscores = batch_zscores.with_columns(
    pl.Series("pval_parametric", pvals_parametric),
    pl.Series("pval_empirical", pvals_empirical)
)

In [ ]:
# Apply FDR correction (Benjamini-Hochberg)
def apply_fdr_correction(pvals):
    """Apply BH FDR correction, handling NaN values."""
    pvals_arr = np.array(pvals, dtype=float)
    mask = ~np.isnan(pvals_arr)
    corrected = np.full(len(pvals_arr), np.nan)
    
    if mask.sum() > 0:
        _, pvals_corr, _, _ = multipletests(pvals_arr[mask], alpha=0.05, method="fdr_bh")
        corrected[mask] = pvals_corr
    
    return corrected

# Apply FDR correction to both p-value types
batch_zscores = batch_zscores.with_columns(
    pl.Series("pval_parametric_fdr", apply_fdr_correction(batch_zscores["pval_parametric"].to_list())),
    pl.Series("pval_empirical_fdr", apply_fdr_correction(batch_zscores["pval_empirical"].to_list()))
)

# Add direction column
batch_zscores = batch_zscores.with_columns(
    pl.when(pl.col("final_zscore") > 0)
    .then(pl.lit("increased"))
    .when(pl.col("final_zscore") < 0)
    .then(pl.lit("decreased"))
    .otherwise(pl.lit("unchanged"))
    .alias("direction")
)

## 7. Results Summary

In [ ]:
# Final output schema
final_results = batch_zscores.select([
    pl.col("Metadata_gene_allele").alias("gene_variant"),
    "Gene",
    pl.col("Metadata_Bio_Batch").alias("bio_batch"),
    "final_zscore",
    "zscore_plate_variability",
    "median_log2FC",
    "n_plates",
    "total_cells",
    "direction",
    "pval_parametric",
    "pval_empirical",
    "pval_parametric_fdr",
    "pval_empirical_fdr"
]).sort(["Gene", "gene_variant", "bio_batch"])

print(f"Final results: {final_results.shape[0]} variant-batch combinations")
final_results.head(10)

In [ ]:
# Summary statistics
total = final_results.shape[0]

print("=== Direction Summary ===")
n_increased = final_results.filter(pl.col("direction") == "increased").shape[0]
n_decreased = final_results.filter(pl.col("direction") == "decreased").shape[0]
print(f"Increased: {n_increased} ({n_increased/total*100:.1f}%)")
print(f"Decreased: {n_decreased} ({n_decreased/total*100:.1f}%)")

print("\n=== Significance (Parametric FDR < 0.05) ===")
sig_param = final_results.filter(pl.col("pval_parametric_fdr") < 0.05)
sig_inc_param = sig_param.filter(pl.col("direction") == "increased").shape[0]
sig_dec_param = sig_param.filter(pl.col("direction") == "decreased").shape[0]
print(f"Significant increased: {sig_inc_param} ({sig_inc_param/total*100:.1f}%)")
print(f"Significant decreased: {sig_dec_param} ({sig_dec_param/total*100:.1f}%)")

print("\n=== Significance (Empirical FDR < 0.05) ===")
sig_emp = final_results.filter(pl.col("pval_empirical_fdr") < 0.05)
sig_inc_emp = sig_emp.filter(pl.col("direction") == "increased").shape[0]
sig_dec_emp = sig_emp.filter(pl.col("direction") == "decreased").shape[0]
print(f"Significant increased: {sig_inc_emp} ({sig_inc_emp/total*100:.1f}%)")
print(f"Significant decreased: {sig_dec_emp} ({sig_dec_emp/total*100:.1f}%)")

## 8. Visualizations

In [ ]:
# Z-score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Z-score histogram
zscores = final_results.filter(pl.col("final_zscore").is_not_null())["final_zscore"].to_numpy()
axes[0].hist(zscores, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(x=0, color='red', linestyle='--', lw=2)
axes[0].axvline(x=-2, color='orange', linestyle=':', lw=1.5, label='|Z|=2')
axes[0].axvline(x=2, color='orange', linestyle=':', lw=1.5)
axes[0].set_xlabel('Final Z-score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Variant Z-scores')
axes[0].legend()

# Volcano plot
pvals = final_results.filter(pl.col("pval_parametric_fdr").is_not_null())
log10_pval = -np.log10(pvals["pval_parametric_fdr"].to_numpy())
zscore_vals = pvals["final_zscore"].to_numpy()

# Color by significance
colors = ['red' if p < 0.05 else 'grey' for p in pvals["pval_parametric_fdr"].to_list()]
axes[1].scatter(zscore_vals, log10_pval, c=colors, alpha=0.5, s=20)
axes[1].axhline(y=-np.log10(0.05), color='red', linestyle='--', lw=1, label='FDR=0.05')
axes[1].axvline(x=0, color='grey', linestyle='-', lw=0.5)
axes[1].set_xlabel('Final Z-score')
axes[1].set_ylabel('-log10(FDR-adjusted p-value)')
axes[1].set_title('Volcano Plot')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# P-value comparison: parametric vs empirical
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

valid_pvals = final_results.filter(
    pl.col("pval_parametric").is_not_null() & 
    pl.col("pval_empirical").is_not_null()
)

ax.scatter(
    -np.log10(valid_pvals["pval_parametric"].to_numpy()),
    -np.log10(valid_pvals["pval_empirical"].to_numpy()),
    alpha=0.5, s=20
)
max_val = max(
    -np.log10(valid_pvals["pval_parametric"].min()),
    -np.log10(valid_pvals["pval_empirical"].min())
)
ax.plot([0, max_val], [0, max_val], 'r--', lw=1)
ax.set_xlabel('-log10(Parametric p-value)')
ax.set_ylabel('-log10(Empirical p-value)')
ax.set_title('Parametric vs Empirical P-values')

plt.tight_layout()
plt.show()

## 9. Comparison with Existing Well-Aggregated Method

Load and compare results from the previous paired t-test approach.

In [ ]:
# Load previous results
prev_results_path = f"{CC_ABUND_DIR}/prot_abundance_diff_summary.csv"

if os.path.exists(prev_results_path):
    prev_results = pl.read_csv(prev_results_path)
    print(f"Loaded {prev_results.shape[0]} previous results")
    
    # Merge for comparison
    comparison = final_results.join(
        prev_results.rename({
            "Gene": "Gene_prev",
            "Bio_Batch": "bio_batch"
        }),
        on=["gene_variant", "bio_batch"],
        how="inner"
    )
    
    print(f"Overlapping variants: {comparison.shape[0]}")
else:
    print(f"Previous results not found at {prev_results_path}")
    comparison = None

In [ ]:
if comparison is not None and comparison.shape[0] > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Correlation between t-statistic and Z-score
    valid_comp = comparison.filter(
        pl.col("final_zscore").is_not_null() &
        pl.col("U2OS_prot_abun_t_stat").is_not_null()
    )
    
    t_stats = valid_comp["U2OS_prot_abun_t_stat"].to_numpy()
    z_scores = valid_comp["final_zscore"].to_numpy()
    
    axes[0].scatter(t_stats, z_scores, alpha=0.5, s=20)
    axes[0].set_xlabel('Paired t-statistic (previous method)')
    axes[0].set_ylabel('Single-cell Z-score (new method)')
    axes[0].set_title(f'Method Comparison (n={len(t_stats)})')
    
    # Add correlation
    corr = np.corrcoef(t_stats, z_scores)[0, 1]
    axes[0].annotate(f'r = {corr:.3f}', xy=(0.05, 0.95), xycoords='axes fraction', fontsize=12)
    
    # Concordance of significant hits
    # Define significance thresholds
    sig_prev = valid_comp.filter(pl.col("U2OS_prot_abun_t_pval_fdr_bh_adj") < 0.05)
    sig_new = valid_comp.filter(pl.col("pval_parametric_fdr") < 0.05)
    
    # Venn diagram data
    sig_prev_set = set(sig_prev["gene_variant"].to_list())
    sig_new_set = set(sig_new["gene_variant"].to_list())
    
    only_prev = len(sig_prev_set - sig_new_set)
    only_new = len(sig_new_set - sig_prev_set)
    both = len(sig_prev_set & sig_new_set)
    
    # Bar chart for concordance
    categories = ['Only Previous', 'Both', 'Only New']
    counts = [only_prev, both, only_new]
    axes[1].bar(categories, counts, color=['lightcoral', 'lightgreen', 'steelblue'])
    axes[1].set_ylabel('Number of variants')
    axes[1].set_title('Concordance of Significant Hits (FDR < 0.05)')
    
    for i, v in enumerate(counts):
        axes[1].text(i, v + 1, str(v), ha='center', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nConcordance statistics:")
    print(f"  Significant in previous only: {only_prev}")
    print(f"  Significant in both: {both}")
    print(f"  Significant in new only: {only_new}")
    
    if len(sig_prev_set | sig_new_set) > 0:
        jaccard = both / len(sig_prev_set | sig_new_set)
        print(f"  Jaccard similarity: {jaccard:.3f}")

## 10. Save Results

In [ ]:
# Save main results
output_path = f"{OUTPUT_DIR}/prot_abundance_singlecell_zscore.csv"
final_results.write_csv(output_path)
print(f"Saved results to {output_path}")

# Save plate-level results for detailed analysis
plate_output_path = f"{OUTPUT_DIR}/prot_abundance_singlecell_zscore_plate_level.csv"
plate_zscores.write_csv(plate_output_path)
print(f"Saved plate-level results to {plate_output_path}")

# Save WT-to-WT Z-scores for reference
wt_output_path = f"{OUTPUT_DIR}/prot_abundance_wt_to_wt_zscores.csv"
pl.DataFrame({"wt_zscore": wt_zscores}).write_csv(wt_output_path)
print(f"Saved WT-to-WT Z-scores to {wt_output_path}")

In [ ]:
# Save 1% screen subset (excluding CCM2 and B_18-19)
oneperc_results = final_results.filter(
    (pl.col("Gene") != "CCM2") &
    (pl.col("bio_batch") != "B_18-19")
)

oneperc_output_path = f"{OUTPUT_DIR}/prot_abundance_singlecell_zscore_oneperc.csv"
oneperc_results.write_csv(oneperc_output_path)
print(f"Saved 1% screen results ({oneperc_results.shape[0]} entries) to {oneperc_output_path}")

## 11. Sanity Checks

In [ ]:
# Sanity check 1: WT-to-WT Z-scores should center around 0
print("=== Sanity Check: WT-to-WT Z-scores ===")
print(f"Mean: {np.mean(wt_zscores):.4f} (expected: ~0)")
print(f"Std:  {np.std(wt_zscores):.4f} (expected: ~1)")
print(f"Proportion |Z| > 2: {np.mean(np.abs(wt_zscores) > 2):.4f} (expected: ~0.05)")

In [ ]:
# Sanity check 2: Check a few known genes with expected behavior
print("\n=== Sample Results ===")
sample_genes = final_results.select("Gene").unique().head(5)["Gene"].to_list()
for gene in sample_genes:
    gene_results = final_results.filter(pl.col("Gene") == gene)
    print(f"\n{gene}: {gene_results.shape[0]} variants")
    display(gene_results.select(["gene_variant", "bio_batch", "final_zscore", "pval_parametric_fdr", "direction"]))

In [ ]:
print("\n=== Analysis Complete ===")